[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/pytorch-lab/blob/main/03_pytorch_regression.ipynb)

# 03 - Регрессия на PyTorch: Auto MPG

**Цель.** Пройти полный цикл обучения регрессионной модели на реальном табличном датасете Auto MPG.

**Результаты обучения:**
- загружать и очищать табличные данные
- кодировать категориальные признаки и стандартизировать числовые
- создавать TensorDataset и DataLoader
- обучать MLP для регрессии
- оценивать RMSE и сравнивать прогнозы с истинными значениями

## Источники
- [Heaton: PyTorch regression](https://github.com/jeffheaton/app_deep_learning/blob/main/t81_558_class_02_2_pytorch_neural.ipynb)

Импортируем библиотеки для работы с данными, обучения модели и визуализации. Фиксируем случайные зерна.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)

Загрузим Auto MPG напрямую из UCI. Признак `horsepower` содержит пропуски в виде `?`, поэтому превращаем их в `NaN` и удаляем такие строки.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
columns = [
    "mpg", "cylinders", "displacement", "horsepower", "weight",
    "acceleration", "model_year", "origin", "car_name",
]
df = pd.read_csv(url, names=columns, sep=r"\s+", na_values="?", comment="\t")
df = df.dropna().reset_index(drop=True)

print(df.shape)
df.head()

Целевая переменная - `mpg`. Название автомобиля удаляем, а категориальный признак `origin` кодируем one-hot представлением.

In [ ]:
target = df["mpg"].astype("float32").to_numpy().reshape(-1, 1)
features = df.drop(columns=["mpg", "car_name"])
features = pd.get_dummies(features, columns=["origin"], prefix="origin", dtype=float)

print(features.head())
print("Number of features:", features.shape[1])

Разделим данные на train и validation. `StandardScaler` обучаем только на train, чтобы не переносить информацию из validation в обучение.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    features, target, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_val_scaled = scaler.transform(X_val).astype("float32")

print("Train shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)

Преобразуем массивы в тензоры и создадим загрузчики данных. Мини-батчи делают цикл обучения ближе к реальной практике.

In [ ]:
train_ds = TensorDataset(torch.from_numpy(X_train_scaled), torch.from_numpy(y_train))
val_ds = TensorDataset(torch.from_numpy(X_val_scaled), torch.from_numpy(y_val))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

Опишем регрессионную MLP через наследование от `nn.Module`: два скрытых слоя с ReLU и один выход без активации.

In [ ]:
class MPGRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 25),
            nn.ReLU(),
            nn.Linear(25, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x):
        return self.net(x)

model = MPGRegressor(X_train_scaled.shape[1])
model

Обучим модель 100 эпох с `MSELoss` и `Adam`. После каждой эпохи считаем среднюю ошибку на train и validation.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_losses, val_losses = [], []

for epoch in range(100):
    model.train()
    batch_losses = []
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_batch_losses = [criterion(model(xb), yb).item() for xb, yb in val_loader]

    train_losses.append(np.mean(batch_losses))
    val_losses.append(np.mean(val_batch_losses))

print(f"Final train MSE: {train_losses[-1]:.2f}")
print(f"Final validation MSE: {val_losses[-1]:.2f}")

График loss помогает увидеть, как меняется качество модели на обучении и валидации. *

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="validation")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Оценим RMSE на validation и выведем таблицу из 10 случайных прогнозов. *

In [ ]:
model.eval()
with torch.no_grad():
    X_val_tensor = torch.from_numpy(X_val_scaled)
    y_val_tensor = torch.from_numpy(y_val)
    val_predictions = model(X_val_tensor)
    rmse = torch.sqrt(nn.MSELoss()(val_predictions, y_val_tensor)).item()

print(f"Validation RMSE: {rmse:.2f} MPG")

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(y_val), size=10, replace=False)
comparison = pd.DataFrame({
    "true_mpg": y_val[sample_idx].ravel(),
    "predicted_mpg": val_predictions.numpy()[sample_idx].ravel(),
})
comparison["absolute_error"] = (comparison["true_mpg"] - comparison["predicted_mpg"]).abs()
comparison.round(2)

## Сравнение топологий

Для отчета полезно сравнить несколько MLP-архитектур на одном и том же разбиении данных. Ниже обучаются три топологии: базовая, более широкая и более глубокая. *

In [ ]:
class FlexibleMPGRegressor(nn.Module):
    def __init__(self, n_features, hidden_layers):
        super().__init__()
        layers = []
        in_features = n_features
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            in_features = hidden_size
        layers.append(nn.Linear(in_features, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def rmse_for_loader(model, loader):
    model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            predictions.append(model(xb))
            targets.append(yb)
    predictions = torch.cat(predictions)
    targets = torch.cat(targets)
    return torch.sqrt(nn.MSELoss()(predictions, targets)).item()


def train_topology(hidden_layers, epochs=100):
    torch.manual_seed(42)
    topology_model = FlexibleMPGRegressor(X_train_scaled.shape[1], hidden_layers)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(topology_model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        topology_model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(topology_model(xb), yb)
            loss.backward()
            optimizer.step()

    return {
        "topology": f"{X_train_scaled.shape[1]}-" + "-".join(map(str, hidden_layers)) + "-1",
        "epochs": epochs,
        "train_rmse": rmse_for_loader(topology_model, train_loader),
        "val_rmse": rmse_for_loader(topology_model, val_loader),
    }

Запустим одинаковый цикл обучения для трех вариантов. В таблице ниже можно взять значения для раздела отчета о результатах регрессии на Auto MPG.

In [ ]:
topologies = [
    [25, 10],
    [50, 25],
    [50, 25, 10],
]

topology_results = pd.DataFrame([train_topology(layers) for layers in topologies])
topology_results.round(3)

> Материал основан на курсе Jeff Heaton T81-558 и книге Maxim Lapan 'Deep Reinforcement Learning Hands-On', глава 3. Адаптировано для учебных целей.